# Step 3. 산불발생 선행기상 및 국지임계치 심화분석

직전 24·48·72시간과 D-1~D-3 선행 기상, 번지유형, 캐나다 지수 분석에 필요한 자료를 준비한다.

## 현재 구현 범위

이 노트북은 **원천 데이터 경로 확인, 로딩, 필수 스키마 검증, 시간 파싱,
기본 행 수 감사**까지 구현한다. 통계 분석과 시각화는 이후 셀에서 이어서 작성한다.

공통 데이터 계약은 `README.md`를 따른다. Step 2~6에서 캐나다 지수를 사건 시각에
결합할 때는 12시 이전이면 전일 정오, 12시 이후이면 당일 정오 자료만 사용한다.

결과 해석은 이 노트북에 작성하지 않는다. 결과표와 플롯을 함께 검토한 해석,
한계와 다음 코드 반영사항은 대응 `진행예정로그.md`에만 기록한다.

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
candidates = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
REPO_ROOT = next(
    (path for path in candidates if (path / "jsw/강원_재_EDA/re_eda_common.py").exists()),
    Path(r"D:/farm-system-public-02"),
)
MODULE_DIR = REPO_ROOT / "jsw/강원_재_EDA"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from re_eda_common import (
    DATA_PATHS,
    DERIVED_WEATHER_COLUMNS,
    RAW_WEATHER_COLUMNS,
    add_canadian_asof_keys,
    check_sources,
    configure_notebook,
    frame_inventory,
    load_access_lines,
    load_canadian_indices,
    load_dem_metadata,
    load_fire,
    load_grid_bundle,
    load_hourly_weather,
    load_infrastructure,
    load_landcover,
    load_roads,
    load_terrain,
)

configure_notebook()

REPO_ROOT: D:\farm-system-public-02
matplotlib font.family: ['Malgun Gothic']


## 1. 원천 파일 존재 여부

In [2]:
SOURCE_KEYS = ['fire', 'weather_cells', 'weather_grid', 'weather_hourly_derived', 'climate_type', 'canada_ffmc', 'canada_fwi']
source_audit = check_sources(SOURCE_KEYS)
display(source_audit)

,데이터키,존재,크기_MB,경로
0,fire,True,0.39,D:\farm-system-public-02\data\강원도_데이터\강원도_산불발생...
1,weather_cells,True,0.05,D:\farm-system-public-02\data\강원도_날씨데이터\강원도날씨_...
2,weather_grid,True,0.09,D:\farm-system-public-02\data\강원도_날씨데이터\강원도날씨_...
3,weather_hourly_derived,True,438.45,D:\farm-system-public-02\data\학습데이터\기상_시간단위_파생...
4,climate_type,True,0.00,D:\farm-system-public-02\data\강원도_날씨데이터\강원도날씨_...
5,canada_ffmc,True,6.75,D:\farm-system-public-02\data\학습데이터\캐나다_FFMC_선...
6,canada_fwi,True,10.10,D:\farm-system-public-02\data\학습데이터\캐나다_FWI_일단...


## 2. 산불 및 기상격자 로딩

In [3]:
fire, fire_points = load_fire()
grid_bundle = load_grid_bundle()
weather_cells = grid_bundle["cells"]
climate_type = grid_bundle["climate"]
weather_grid = grid_bundle["grid"]

print("산불:", len(fire), "건")
print("좌표 유효 산불:", len(fire_points), "건")
print("발생시각 결측:", int(fire["기준시각"].isna().sum()), "건")
display(fire.head())

산불: 3405 건
좌표 유효 산불: 3405 건
발생시각 결측: 0 건


,fire_id,연도,월,일,시간,진화소요시간(HH),위도,경도,발생지역시도명,발생지역시군구명,발생지역읍면동명,발생지역리명,발생지역번지,발생원인명,피해면적(ha),피해금액,발생일시,기준시각,발생날짜,번지유형
0,F_003556,2020,4,30,18,NaN,37.432421,127.803653,강원특별자치도,원주시,지정면,월송리,산172,NaN,NaN,NaN,2020-04-30 18:00:00,2020-04-30 18:00:00,2020-04-30,임야번지(산)
1,F_006688,2020,4,30,18,NaN,37.432421,127.803653,강원특별자치도,원주시,지정면,월송리,산172,NaN,NaN,NaN,2020-04-30 18:00:00,2020-04-30 18:00:00,2020-04-30,임야번지(산)
2,F_006690,2020,5,1,22,NaN,37.773694,127.446131,강원특별자치도,고성군,토성면,도원리,NaN,NaN,NaN,NaN,2020-05-01 22:00:00,2020-05-01 22:00:00,2020-05-01,일반번지
3,F_007615,2020,1,1,12,NaN,37.761451,128.841125,강원특별자치도,강릉시,성산면,위촌리,877,NaN,NaN,NaN,2020-01-01 12:00:00,2020-01-01 12:00:00,2020-01-01,일반번지
4,F_007616,2020,1,3,12,NaN,37.329735,129.043221,강원특별자치도,삼척시,신기면,대이리,30,NaN,NaN,NaN,2020-01-03 12:00:00,2020-01-03 12:00:00,2020-01-03,일반번지


## 3. 누수 방지 시간기상 파생자료 로딩

In [4]:
hourly_weather = load_hourly_weather(
    derived=True,
    columns=DERIVED_WEATHER_COLUMNS,
)
print("기간:", hourly_weather["일시"].min(), "~", hourly_weather["일시"].max())
print("기상셀 수:", hourly_weather["기상셀ID"].nunique())
display(hourly_weather.head())

기간: 2020-01-01 00:00:00 ~ 2021-12-31 23:00:00
기상셀 수: 92


,기상셀ID,일시,시점_기온_C,시점_풍속_m_s,시점_습도_pct,직전24h_평균풍속,직전24h_최대풍속,직전48h_평균풍속,직전48h_최대풍속,직전24h_평균기온_C,직전24h_평균습도,직전24h_최소습도,직전48h_평균습도,직전48h_최소습도,직전24h_강수량합,직전48h_강수량합,시점_풍향_deg,풍향_sin,풍향_cos,서풍계열_여부,시점_현지기압_hPa,시점_해면기압_hPa,기압변동_3h,D-1_최소습도_pct,D-1_평균습도_pct,D-1_강수량합_mm,D-2_최소습도_pct,D-3_최소습도_pct
0,YD_0001,2020-01-01 00:00:00,-4.18,4.52,29.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,226.26,-0.722485,-0.691387,1,999.82,1021.12,NaN,NaN,NaN,NaN,NaN,NaN
1,YD_0001,2020-01-01 01:00:00,-3.59,4.28,40.85,4.520,4.52,4.520,4.52,-4.1800,29.0000,29.0,29.0000,29.0,0.0,0.0,235.90,-0.828060,-0.560639,1,999.68,1020.90,NaN,NaN,NaN,NaN,NaN,NaN
2,YD_0001,2020-01-01 02:00:00,-2.97,4.01,44.09,4.400,4.52,4.400,4.52,-3.8850,34.9250,29.0,34.9250,29.0,0.0,0.0,243.05,-0.891402,-0.453213,1,999.70,1020.91,NaN,NaN,NaN,NaN,NaN,NaN
3,YD_0001,2020-01-01 03:00:00,-2.79,4.89,50.15,4.270,4.52,4.270,4.52,-3.5800,37.9800,29.0,37.9800,29.0,0.0,0.0,233.54,-0.804272,-0.594261,1,999.58,1020.75,-0.24,NaN,NaN,NaN,NaN,NaN
4,YD_0001,2020-01-01 04:00:00,-2.54,4.32,52.23,4.425,4.89,4.425,4.89,-3.3825,41.0225,29.0,41.0225,29.0,0.0,0.0,245.21,-0.907851,-0.419294,1,999.42,1020.55,-0.26,NaN,NaN,NaN,NaN,NaN


## 4. 캐나다 지수 및 사건별 as-of 키 준비

In [5]:
canadian_indices = load_canadian_indices()
fire = add_canadian_asof_keys(fire, time_column="기준시각")

asof_violations = (
    fire["캐나다지수_기준시각"] > fire["기준시각"]
).sum()
assert asof_violations == 0
display(
    fire[
        [
            "fire_id",
            "기준시각",
            "캐나다지수_기준날짜",
            "캐나다지수_기준시각",
            "캐나다지수_시차시간",
            "캐나다지수_당일사용여부",
        ]
    ].head()
)

,fire_id,기준시각,캐나다지수_기준날짜,캐나다지수_기준시각,캐나다지수_시차시간,캐나다지수_당일사용여부
0,F_003556,2020-04-30 18:00:00,2020-04-30,2020-04-30 12:00:00,6.0,True
1,F_006688,2020-04-30 18:00:00,2020-04-30,2020-04-30 12:00:00,6.0,True
2,F_006690,2020-05-01 22:00:00,2020-05-01,2020-05-01 12:00:00,10.0,True
3,F_007615,2020-01-01 12:00:00,2020-01-01,2020-01-01 12:00:00,0.0,True
4,F_007616,2020-01-03 12:00:00,2020-01-03,2020-01-03 12:00:00,0.0,True


## 5. 선행기상 컬럼 가용성 감사

In [6]:
expected_lead_columns = [
    "직전24h_평균풍속",
    "직전24h_최대풍속",
    "직전48h_평균풍속",
    "직전48h_최대풍속",
    "직전24h_평균습도",
    "직전24h_최소습도",
    "직전48h_평균습도",
    "직전48h_최소습도",
    "직전24h_강수량합",
    "직전48h_강수량합",
    "D-1_최소습도_pct",
    "D-1_평균습도_pct",
    "D-1_강수량합_mm",
    "D-2_최소습도_pct",
    "D-3_최소습도_pct",
]
display(hourly_weather[expected_lead_columns].isna().mean().rename("결측률").to_frame())
print("주의: 직전72h 및 추가 D-2/D-3 집계는 원본 시간자료에서 누수 없이 후속 계산합니다.")

,결측률
직전24h_평균풍속,0.000057
직전24h_최대풍속,0.000057
직전48h_평균풍속,0.000057
직전48h_최대풍속,0.000057
직전24h_평균습도,0.000057
직전24h_최소습도,0.000057
직전48h_평균습도,0.000057
직전48h_최소습도,0.000057
직전24h_강수량합,0.000057
직전48h_강수량합,0.000057


주의: 직전72h 및 추가 D-2/D-3 집계는 원본 시간자료에서 누수 없이 후속 계산합니다.


## 로딩 결과 요약

In [7]:
loaded_frames = {
    "fire": fire,
    "fire_points": fire_points,
    "weather_cells": weather_cells,
    "climate_type": climate_type,
    "weather_grid": weather_grid,
    "hourly_weather": hourly_weather,
    "canadian_indices": canadian_indices,
}
display(frame_inventory(loaded_frames))

,이름,행,열,메모리_MB,CRS
0,fire,3405,25,0.86,NaN
1,fire_points,3405,21,0.78,EPSG:4326
2,weather_cells,92,12,0.05,NaN
3,climate_type,92,3,0.00,NaN
4,weather_grid,92,12,0.01,EPSG:4326
5,hourly_weather,1614048,28,355.57,NaN
6,canadian_indices,67252,16,8.66,NaN


## 다음 구현 범위

직전72시간·D-1~D-3 파생을 보강하고, 매칭 대조군 기준 상대 분위수와 지역·번지별 임계치를 계산한다.

현재 노트북은 로딩과 입력 감사까지만 실행한다. 이후 분석 셀에서도
`README.md`의 미래 정보 누수 방지 규칙과 대조군 정의를 유지해야 한다.
실행 결과의 해석은 대응 `진행예정로그.md`에 작성한다.